In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from src.shared_preprocessing import RANDOM_STATE, make_cv

SEED = RANDOM_STATE

models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {"model__C": [0.01, 0.1, 1, 10], "model__class_weight": [None, "balanced"]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED),
        {
            "model__max_depth": [None, 5, 10, 20],
            "model__min_samples_leaf": [1, 5, 20],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=1),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 5],
            "model__class_weight": [None, "balanced"],
        },
    ),
}

In [ ]:
def run_base_models(preprocessor, X_train, y_train, *, scoring):
    """Compare baseline and tuned models using training-only cross-validation."""
    cv = make_cv()
    summary = []
    tuned_models = {}

    for name, (estimator, grid) in models.items():
        pipeline = Pipeline([
            ("preprocess", clone(preprocessor)),
            ("model", clone(estimator)),
        ])

        baseline_cv = cross_val_score(
            pipeline,
            X_train,
            y_train,
            scoring=scoring,
            cv=cv,
            error_score="raise",
        ).mean()

        search = GridSearchCV(
            pipeline,
            grid,
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_

        summary.append({
            "model": name,
            "baseline_cv_mean": baseline_cv,
            "tuned_cv_mean": search.best_score_,
            "cv_change": search.best_score_ - baseline_cv,
            "best_parameters": search.best_params_,
        })

    return pd.DataFrame(summary), tuned_models

In [ ]:
from src.shared_preprocessing import PRIMARY_SCORING, build_preprocessor, get_split

# The test split is not evaluated in this notebook.
# It is reserved for the final comparison after ensemble selection.
X_train, _, y_train, _ = get_split()
preprocessor = build_preprocessor(X_train)

tuning_summary, tuned_models = run_base_models(
    preprocessor,
    X_train,
    y_train,
    scoring=PRIMARY_SCORING,
)
display(tuning_summary)